# Phase 3 — Author Screening

Establishes which authors can support an event study before their full
publication histories are extracted.

**Input:** `data/interim/phase02_author_paper.csv`, the OpenAlex snapshot

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase03_authors_screened.csv` | every author, with screening flags |
| `data/interim/phase04_author_queue.csv` | those passing, as the Phase 4 input |

An author whose observable record does not reach several years either side of
the retraction cannot contribute a pre-period trend or a post-period response.
Screening here costs one pass over the `authors` entity; extracting histories
for authors who cannot support an estimate would cost a pass over `works`.

An author appearing on several retracted papers is anchored on the earliest:
the first retraction is the first public signal, and later ones fall inside the
observation window rather than defining it.

## Career span

Career start and end are taken from `counts_by_year` on the `authors` entity.
This field covers a rolling window rather than an author's full history, so it
is a lower bound on career length. It is adequate for a screening threshold and
for the coarse career bands used in matching, but it is not the author's career
start and is not used as a continuous covariate. Phase 8 recomputes career
stage from the extracted publication histories, where the true first year is
observable.

In [1]:
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from snapshot import Snapshot, concat, strip_id

AUTHOR_PAPER = "data/interim/phase02_author_paper.csv"
OUT_SCREENED = "data/interim/phase03_authors_screened.csv"
OUT_QUEUE = "data/interim/phase04_author_queue.csv"

DATA_HORIZON = 2025

MIN_TOTAL_WORKS = 3
MIN_PRE_YEARS = 3
MIN_POST_YEARS = 3

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

# Restrict the scan for testing. None runs the full entity.
LIMIT_FILES = None
SKIP_FILES = 0

pd.set_option("display.width", 200)
os.makedirs("data/interim", exist_ok=True)

print(f"criteria: found in OpenAlex; at least {MIN_TOTAL_WORKS} works; "
      f"at least {MIN_PRE_YEARS} pre-years;")
print(f"          at least {MIN_POST_YEARS} post-years against a horizon of "
      f"{DATA_HORIZON}")

snap = Snapshot()
d = snap.describe("authors")
print(f"\nauthors: {d['files']:,} files, {d['bytes'] / 2**30:.0f} GiB, "
      f"{d['partitions']:,} partitions")

criteria: found in OpenAlex; at least 3 works; at least 3 pre-years;
          at least 3 post-years against a horizon of 2025

authors: 1,961 files, 49 GiB, 1,664 partitions


## One row per author

An author on several retracted papers appears several times in the Phase 2
table. Collapsing to the earliest retraction gives each author a single event
date, and records how many retractions they carry and whether those span more
than one attribution category.

In [2]:
ap = pd.read_csv(AUTHOR_PAPER, low_memory=False)
ap["retraction_date"] = pd.to_datetime(ap.retraction_date, errors="coerce")
ap["author_id"] = ap.author_id.astype(str)

print(f"author-paper rows {len(ap):,}")
print(f"unique authors    {ap.author_id.nunique():,}")

first = (ap.sort_values("retraction_date")
           .groupby("author_id", as_index=False)
           .first()
           .rename(columns={"retraction_year": "first_retraction_year",
                            "category": "first_category",
                            "position": "first_position"}))

counts = ap.groupby("author_id").agg(
    n_retractions=("openalex_id", "nunique"),
    n_categories=("category", "nunique"),
).reset_index()
first = first.merge(counts, on="author_id")

print(f"\nretractions per author")
print(first.n_retractions.value_counts().sort_index().head(6).to_string())
print(f"\n  more than one:                {int((first.n_retractions > 1).sum()):,} "
      f"({(first.n_retractions > 1).mean():.1%})")
print(f"  spanning several categories:  {int((first.n_categories > 1).sum()):,}")

author_ids = set(first.author_id)

author-paper rows 78,889
unique authors    63,758

retractions per author
n_retractions
1    56417
2     4890
3     1192
4      483
5      283
6      140

  more than one:                7,341 (11.5%)
  spanning several categories:  2,250


## Scan

One pass over the `authors` entity, retaining records for authors on a
retracted paper.

In [3]:
AUTHOR_COLS = ["id", "display_name", "works_count", "cited_by_count",
               "counts_by_year", "last_known_institutions"]

want = pa.array(sorted(author_ids | {"https://openalex.org/" + a
                                     for a in author_ids}), type=pa.string())


def handler(tbl, path):
    col = tbl.column("id")
    if isinstance(col, pa.ChunkedArray):
        col = col.combine_chunks()
    hit = pc.is_in(col, value_set=want)
    rows = pc.indices_nonzero(pc.fill_null(hit, False))
    return tbl.take(rows) if len(rows) else None


t0 = time.time()
res = snap.scan("authors", AUTHOR_COLS, handler,
                limit_files=LIMIT_FILES, skip_files=SKIP_FILES,
                progress_every=200)
authors_tbl = concat(res)
print(f"\nelapsed {(time.time() - t0) / 60:.1f} min")
print(f"author records found: "
      f"{authors_tbl.num_rows if authors_tbl is not None else 0:,} "
      f"of {len(author_ids):,}")

scanning authors: 1,961 files, 49.1 GiB on disk
  projecting 6 of 20 columns
  200/1,961 files | 7.1M rows | kept 1,082 | 132 MiB/s | eta 6m
  400/1,961 files | 10.5M rows | kept 1,284 | 119 MiB/s | eta 7m
  600/1,961 files | 10.7M rows | kept 1,289 | 104 MiB/s | eta 8m
  800/1,961 files | 11.0M rows | kept 1,297 | 91 MiB/s | eta 9m
  1,000/1,961 files | 11.3M rows | kept 1,300 | 83 MiB/s | eta 9m
  1,200/1,961 files | 11.5M rows | kept 1,303 | 77 MiB/s | eta 10m
  1,400/1,961 files | 11.7M rows | kept 1,305 | 72 MiB/s | eta 11m
  1,600/1,961 files | 65.8M rows | kept 3,616 | 192 MiB/s | eta 3m
  1,800/1,961 files | 79.5M rows | kept 3,935 | 189 MiB/s | eta 3m
  done: 119,129,660 rows scanned, 63,757 kept, 2.9 min

elapsed 2.9 min
author records found: 63,757 of 63,758


## Screening table

In [4]:
if authors_tbl is None or not authors_tbl.num_rows:
    raise SystemExit("no author records found")

cby = authors_tbl.column("counts_by_year").to_pylist()
insts = authors_tbl.column("last_known_institutions").to_pylist()

fetched = pd.DataFrame({
    "author_id": [strip_id(x) for x in authors_tbl.column("id").to_pylist()],
    "works_count": authors_tbl.column("works_count").to_pylist(),
    "cited_by_count": authors_tbl.column("cited_by_count").to_pylist(),
    "first_pub_year": [min((c["year"] for c in r if c.get("year") is not None),
                           default=np.nan) for r in cby],
    "last_pub_year": [max((c["year"] for c in r if c.get("year") is not None),
                          default=np.nan) for r in cby],
    "last_institution": [(i[0].get("display_name") if i else None)
                         for i in insts],
    "last_country": [(i[0].get("country_code") if i else None) for i in insts],
})
fetched = fetched.drop_duplicates("author_id")

sc = first.merge(fetched, on="author_id", how="left")
sc["found"] = sc.works_count.notna()

sc["career_span"] = sc.last_pub_year - sc.first_pub_year
sc["pre_years"] = sc.first_retraction_year - sc.first_pub_year
sc["post_years"] = DATA_HORIZON - sc.first_retraction_year

print(f"found in the snapshot  {int(sc.found.sum()):,} of {len(sc):,} "
      f"({sc.found.mean():.1%})")

print(f"\nworks per author")
print(sc.works_count.describe().round(1).to_string())
print()
for k in [1, 3, 5, 10, 20, 50]:
    n = int((sc.works_count >= k).sum())
    print(f"  at least {k:>2} works  {n:>7,}  ({n / len(sc):>5.1%})")

found in the snapshot  63,757 of 63,758 (100.0%)

works per author
count    63757.0
mean       123.3
std        227.5
min          1.0
25%         16.0
50%         53.0
75%        139.0
max      10898.0

  at least  1 works   63,757  (100.0%)
  at least  3 works   59,379  (93.1%)
  at least  5 works   56,657  (88.9%)
  at least 10 works   52,082  (81.7%)
  at least 20 works   45,838  (71.9%)
  at least 50 works   33,099  (51.9%)


In [5]:
sc["pass_found"] = sc.found
sc["pass_works"] = sc.works_count >= MIN_TOTAL_WORKS
sc["pass_pre"] = sc.pre_years >= MIN_PRE_YEARS
sc["pass_post"] = sc.post_years >= MIN_POST_YEARS
sc["passes"] = sc[["pass_found", "pass_works", "pass_pre",
                   "pass_post"]].all(axis=1)

running = pd.Series(True, index=sc.index)
for label, col in [("found in OpenAlex", "pass_found"),
                   (f"at least {MIN_TOTAL_WORKS} works", "pass_works"),
                   (f"at least {MIN_PRE_YEARS} pre-years", "pass_pre"),
                   (f"at least {MIN_POST_YEARS} post-years", "pass_post")]:
    before = int(running.sum())
    running = running & sc[col]
    print(f"  {label:<26} dropped {before - int(running.sum()):>7,}   "
          f"remaining {int(running.sum()):>7,}")

n_pass = int(sc.passes.sum())
print(f"\npass  {n_pass:,} / {len(sc):,}  ({n_pass / len(sc):.1%})")

  found in OpenAlex          dropped       1   remaining  63,757
  at least 3 works           dropped   4,378   remaining  59,379
  at least 3 pre-years       dropped   3,558   remaining  55,821
  at least 3 post-years      dropped       0   remaining  55,821

pass  55,821 / 63,758  (87.6%)


## Does the screen fall evenly across categories?

The editorial-compromise category is paper-mill heavy, and paper-mill authors
often have thin publication records. A screen that removed that category
disproportionately would weaken the arm the study uses as its
no-attributed-fault comparison.

In [6]:
by_arm = sc.groupby("first_category").agg(
    authors=("author_id", "size"),
    passing=("passes", "sum"),
    rate=("passes", "mean"),
    median_works=("works_count", "median"),
)
by_arm["rate"] = (by_arm["rate"] * 100).round(1)
print(by_arm.to_string())

study = by_arm.loc[[a for a in ARMS if a in by_arm.index], "rate"]
print(f"\nspread across the three study categories: "
      f"{study.max() - study.min():.1f} points")

print(f"\npass rate by byline position")
pos = sc.groupby("first_position").passes.agg(["size", "sum", "mean"])
pos.columns = ["authors", "passing", "rate"]
pos["rate"] = (pos["rate"] * 100).round(1)
print(pos.to_string())

                      authors  passing  rate  median_works
first_category                                            
AUTHOR_MISCONDUCT       34230    29884  87.3          50.0
EDITORIAL_COMPROMISE     9286     8004  86.2          49.0
ETHICS_VIOLATION          926      822  88.8          57.5
HONEST_ERROR            13198    11962  90.6          68.0
UNCLASSIFIED              492      402  81.7          50.5
UNCONFIRMED_CONCERNS     5626     4747  84.4          49.0

spread across the three study categories: 4.4 points

pass rate by byline position
                authors  passing  rate
first_position                        
first             13879    11784  84.9
last              11359     9922  87.3
middle            38520    34115  88.6


## Authors with more than one retraction

An author carrying several retractions is anchored on the first. Later
retractions occur inside the observation window, so their post-period reflects
more than one event. Phase 8 restricts the primary sample to authors with
exactly one.

In [7]:
p = sc[sc.passes]
multi = p[p.n_retractions > 1]

print(f"passing authors            {len(p):,}")
print(f"  one retraction           {int((p.n_retractions == 1).sum()):,} "
      f"({(p.n_retractions == 1).mean():.1%})")
print(f"  more than one            {len(multi):,} ({len(multi) / len(p):.1%})")
print(f"  spanning categories      {int((multi.n_categories > 1).sum()):,} "
      f"({(multi.n_categories > 1).mean():.1%} of those)")

print(f"\nby category")
tab = p.groupby("first_category").agg(
    authors=("author_id", "size"),
    multi=("n_retractions", lambda s: int((s > 1).sum())),
)
tab["share"] = (100 * tab.multi / tab.authors).round(1)
print(tab.to_string())

print(f"\nmedian works by number of retractions")
print(p.groupby(p.n_retractions.clip(upper=4)).works_count.median().to_string())

passing authors            55,821
  one retraction           48,799 (87.4%)
  more than one            7,022 (12.6%)
  spanning categories      2,198 (31.3% of those)

by category
                      authors  multi  share
first_category                             
AUTHOR_MISCONDUCT       29884   4516   15.1
EDITORIAL_COMPROMISE     8004    872   10.9
ETHICS_VIOLATION          822     99   12.0
HONEST_ERROR            11962   1034    8.6
UNCLASSIFIED              402     57   14.2
UNCONFIRMED_CONCERNS     4747    444    9.4

median works by number of retractions
n_retractions
1     64.0
2     88.0
3     99.0
4    125.0


## Write

In [8]:
sc.to_csv(OUT_SCREENED, index=False)
print(f"{OUT_SCREENED}: {len(sc):,} rows, {len(sc.columns)} columns")

QUEUE_COLS = ["author_id", "author_display_name", "first_retraction_year",
              "first_category", "first_position", "n_retractions",
              "n_categories", "works_count", "cited_by_count",
              "first_pub_year", "last_pub_year", "career_span",
              "last_country", "last_institution", "country", "source_id",
              "rw_journal", "rw_publisher"]

queue = sc[sc.passes][[c for c in QUEUE_COLS if c in sc.columns]]
queue.to_csv(OUT_QUEUE, index=False)
print(f"{OUT_QUEUE}: {len(queue):,} rows")

print(f"\nqueue by category")
print(queue.first_category.value_counts().to_string())
print(f"\nin the three study categories: "
      f"{int(queue.first_category.isin(ARMS).sum()):,}")

print(f"\nqueue by retraction year")
print(queue.first_retraction_year.value_counts().sort_index().to_string())

data/interim/phase03_authors_screened.csv: 63,758 rows, 33 columns
data/interim/phase04_author_queue.csv: 55,821 rows

queue by category
first_category
AUTHOR_MISCONDUCT       29884
HONEST_ERROR            11962
EDITORIAL_COMPROMISE     8004
UNCONFIRMED_CONCERNS     4747
ETHICS_VIOLATION          822
UNCLASSIFIED              402

in the three study categories: 49,850

queue by retraction year
first_retraction_year
2015.0     4056
2016.0     4067
2017.0     4194
2018.0     4473
2019.0     5761
2020.0     7340
2021.0    10994
2022.0    14936


## Extraction cost for Phase 4

The queue determines how many author histories Phase 4 retrieves. The works
per author reported here is an estimate from the `authors` entity; the
extraction itself counts what is present in `works`.

In [9]:
total_works = queue.works_count.sum()
print(f"authors queued        {len(queue):,}")
print(f"works to retrieve     {int(total_works):,} (estimated)")
print(f"  median per author   {queue.works_count.median():.0f}")
print(f"  mean per author     {queue.works_count.mean():.1f}")
print(f"  90th percentile     {queue.works_count.quantile(0.9):.0f}")
print(f"  maximum             {int(queue.works_count.max()):,}")

authors queued        55,821
works to retrieve     7,806,704 (estimated)
  median per author   67
  mean per author     139.9
  90th percentile     326
  maximum             10,898

